# Lazypredict

In [ ]:
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

In [ ]:
df = pd.read_excel("../Data Final/Final_PM14-Facial.xlsx")
df.info()

In [ ]:
# Data Preparation

# Cleaning
df['MDWT'] = pd.to_numeric(df['MDWT'], errors='coerce')
df = df.dropna()

# Cleaning - GSM
df['GSM'] = df['GSM'].astype(str)
df = df[df['GSM'].str.len() <= 4].copy()
df['GSM'] = df['GSM'].str.replace(',', '.')
df['GSM'] = df['GSM'].astype(float)

# Create Pseudo_Mass Feature
df['pseudo_mass'] = (df['Mean_Stock Flow'] * df['Mean_Stock Consistency'])/ df['Mean_Yankee Speed']

# Create Coating-Release Ratio Feature
df['coating_release_ratio'] = df['Mean_Flow Coating'] / df['Mean_Flow Release']

# X Variables
features = [
    'Mean_Yankee Pressure',
    'Mean_Creping',
    '% NBKP',
    'Mean_Load KWH Refiner',
    'Mean_Jet Wire Ratio',
    'GSM',
    'coating_release_ratio',
    'pseudo_mass'
]
X = df[features]

# Y Variables
y = df['MDT']

In [ ]:
X.tail()

In [ ]:
y.head()

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

In [ ]:
# Hitung MAPE untuk tiap model
mape_scores = {}

for model_name, y_pred in predictions.items():
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mape_scores[model_name] = mape

# Tambahkan ke tabel hasil
models["MAPE"] = pd.Series(mape_scores)

# Urutkan (semakin kecil semakin baik)
models = models.sort_values(by="MAPE")

In [ ]:
print(models)

# Regressor

In [ ]:
# Import Libraries
import numpy as np

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error)

# Model Dasar
model = ExtraTreesRegressor(random_state=42,n_jobs=1)

# Grid Search Hyper Parameter
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 8, 10, 15],
    'min_samples_split': [2, 4, 6, 10],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 0.5, 0.7],
    'bootstrap': [True, False]
}

# K-Fold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# GGrid Search CV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1,
    verbose=1
)

# Training + Tuning
grid_search.fit(X, y)

# Model Terbaik
best_model = grid_search.best_estimator_
print("Best Parameters:")
print(grid_search.best_params_)

# Prrediksi
y_pred = best_model.predict(X)

# Matrik Evaluasi
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)
epsilon = 1e-8
mape = np.mean(np.abs((y - y_pred) / (y + epsilon))) * 100

# Hasil
print("\n===== HASIL MODEL TERBAIK =====")
print(f"R2    : {r2:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"MAE   : {mae:.4f}")
print(f"MAPE  : {mape:.2f}%")

### Pickle Files

In [ ]:
import joblib

In [ ]:
joblib.dump(model, 'model_PM14-Facial.pkl')

In [ ]:
features_pkl = X.columns.tolist()
joblib.dump(features_pkl, 'features_PM14-Facial.pkl')

### Features Importance

In [ ]:
# Feature Importance
import pandas as pd

feat_imp = pd.DataFrame({
    "Feature": features, 
    "Importance": model.feature_importances_
})

# Urutkan dari terbesar
feat_imp = feat_imp.sort_values(by="Importance", ascending=False)


# Plot
import matplotlib.pyplot as plt
plt.figure()
plt.barh(feat_imp["Feature"], feat_imp["Importance"])
plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance - Extra Tree")

plt.tight_layout()
plt.show()

### SHAP Analysis

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
print(list(X_train.columns))
print(list(X_test.columns))

In [ ]:
print(X_train.isnull().sum())
print(X_test.isnull().sum())

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=10)

In [ ]:
shap.plots.scatter(shap_values[:, "Mean_Load KWH Refiner"])

In [ ]:
shap.plots.scatter(shap_values[:, "pseudo_mass"])

In [ ]:
shap_values = explainer.shap_values(np.array(X_test))
shap.initjs()
i = 0  # index data

shap.force_plot(
    explainer.expected_value,
    shap_values[i],
    X_test.iloc[i]
)